# 第89章 不平衡分类与阈值选择

在类别不平衡任务中使用 PR-AUC、类别权重和阈值选择，使模型评价与漏判、误报和业务容量一致。


## 先解决一个小问题

围绕“不平衡分类与阈值选择”完成一个可验证的小型建模实验：先明确输入和目标，再比较方法带来的变化。在类别不平衡任务中使用 PR-AUC、类别权重和阈值选择，使模型评价与漏判、误报和业务容量一致。


## 这章为什么先学

这是“机器学习”建模主线中的第 89 章，重点放在“不平衡分类与阈值选择”对应的一个具体决策，而不是重复完整流程。


## 开始前确认

- 能够使用 pandas 读取、筛选和汇总数据
- 理解训练集、测试集和基本统计指标
- 本章会进一步练习：识别准确率陷阱、计算 ROC-AUC 与 PR-AUC、使用 class_weight


## 做完要留下什么

完成一份围绕“不平衡分类与阈值选择”的可运行实验：包含数据准备、方法执行、指标或图表证据，以及一句有边界的结论。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 识别准确率陷阱
- 计算 ROC-AUC 与 PR-AUC
- 使用 class_weight
- 按目标召回率或有限容量选择阈值


## 核心概念

- PR-AUC 的基线等于正类比例
- 类别权重改变训练损失而非数据本身
- 阈值决定最终错误成本
- 重采样和阈值必须只依据训练/验证数据设计


## 示例 1：构造不平衡任务

从乳腺癌数据中固定抽取较少正类，演示准确率为何可能误导。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

data = load_breast_cancer(as_frame=True); X=data.data; original=data.target
rng = np.random.default_rng(89)
positive_idx=np.flatnonzero(original.to_numpy()==0)
negative_idx=np.flatnonzero(original.to_numpy()==1)
keep = np.r_[rng.choice(positive_idx,60, replace=False), negative_idx]
X_imb=X.iloc[keep]; y_imb=(original.iloc[keep].to_numpy()==0).astype(int)
X_train, X_test, y_train, y_test=train_test_split(X_imb, y_imb, stratify=y_imb, test_size=.3, random_state=89)
print('正类比例:', round(y_imb.mean(),3), ' 全预测负类准确率:', round(1-y_test.mean(),3))


## 示例 2：权重、概率与阈值

比较普通与平衡权重模型，并按验证目标选择阈值。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rows = []; models={}
for name, weight in [('普通', None), ('平衡', 'balanced')]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight=weight)).fit(X_train, y_train)
    p=m.predict_proba(X_test)[:,1]; pred=p>=.5; models[name]=(m, p)
    rows.append([name, roc_auc_score(y_test, p), average_precision_score(y_test, p), precision_score(y_test, pred), recall_score(y_test, pred)])
display(pd.DataFrame(rows, columns=['模型', 'ROC_AUC', 'PR_AUC', 'precision', 'recall']).set_index('模型').round(3))


## 示例 3：有限容量 Top-K

当人工复核只能覆盖 10% 样本时，直接评价最高分名单。


In [ ]:
prob = models['平衡'][1]
ranked = pd.DataFrame({'y':y_test, 'prob':prob}).sort_values('prob', ascending=False)
k = max(1, int(len(ranked)*.1)); top=ranked.head(k)
lift = top.y.mean()/ranked.y.mean()
print(f'Top 10% 样本={k}, precision={top.y.mean():.3f}, lift={lift:.2f}, recall={top.y.sum()/ranked.y.sum():.3f}')


## 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 常见误区

- 不平衡任务只报告准确率
- 在测试集上选择阈值再报告测试性能
- 使用类别权重后仍假设概率天然校准
- 只追求召回率而不考虑人工容量和误报成本


## 综合练习

1. 在平衡权重模型上寻找召回率至少 0.8 的最高阈值
2. 报告精确率
3. 输出阈值和两个指标

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“在平衡权重模型上寻找召回率至少 0.8 的最高阈值”。
2. **独立完成**：不复制示例代码，完成“报告精确率”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出阈值和两个指标”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
choices = []
for threshold in np.linspace(.05,.95,91):
    pred=prob>=threshold; recall=recall_score(y_test, pred)
    if recall>=.8: choices.append((threshold, precision_score(y_test, pred), recall))
selected = max(choices, key=lambda x:x[0])
print('threshold, precision, recall:', tuple(round(v,3) for v in selected))

# 自检
assert selected[2] >= 0.8
assert 0 < selected[0] < 1


## 本章小结

在类别不平衡任务中使用 PR-AUC、类别权重和阈值选择，使模型评价与漏判、误报和业务容量一致。


### 你已经掌握

- 识别准确率陷阱
- 计算 ROC-AUC 与 PR-AUC
- 使用 class_weight
- 按目标召回率或有限容量选择阈值


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 构造不平衡任务 | 从乳腺癌数据中固定抽取较少正类，演示准确率为何可能误导。 | `np.random.default_rng()`、`np.flatnonzero()`、`original.to_numpy()`、`rng.choice()` |
| 权重、概率与阈值 | 比较普通与平衡权重模型，并按验证目标选择阈值。 | `m.predict_proba()`、`rows.append()`、`pd.DataFrame()`、`.fit()` |
| 有限容量 Top-K | 当人工复核只能覆盖 10% 样本时，直接评价最高分名单。 | `pd.DataFrame()`、`ranked.head()`、`top.y.mean()`、`ranked.y.mean()` |


### 需要注意

- 不平衡任务只报告准确率
- 在测试集上选择阈值再报告测试性能
- 使用类别权重后仍假设概率天然校准
- 只追求召回率而不考虑人工容量和误报成本


### 完成检查

- [ ] 能够识别准确率陷阱
- [ ] 能够计算 ROC-AUC 与 PR-AUC
- [ ] 能够使用 class_weight
- [ ] 能够按目标召回率或有限容量选择阈值


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
